# Wikipedia RfC Baseline Analysis

Run this notebook inside the PAWS environment: https://wikitech.wikimedia.org/wiki/PAWS

Queries the `enwiki` analytics replica for:
- **Q1** Participant tenure distribution in RfC discussions
- **Q2** RfC volume over time (pages + edits per year)
- **Q3** Policy page edit activity
- **Q4** RfA participant tenure (for comparison with RfC)

Outputs CSVs to `data/` for local analysis.

In [ ]:
import wmpaws
import pandas as pd
from pathlib import Path

OUT = Path('rfc-analysis/data')
OUT.mkdir(parents=True, exist_ok=True)
WIKI = 'enwiki'
print(f'Output folder: {OUT.resolve()}')

## Q2 — RfC volume over time

Count distinct RfC pages touched per year/month, and total edit volume.
Use this to understand the overall shape of the dataset before heavier queries.

In [ ]:
df_volume = wmpaws.run_sql("""
SELECT
    LEFT(r.rev_timestamp, 6)       AS ym,
    COUNT(DISTINCT r.rev_page)     AS distinct_pages,
    COUNT(*)                       AS total_edits
FROM revision r
JOIN page p ON r.rev_page = p.page_id
WHERE p.page_namespace = 4
  AND p.page_title LIKE 'Requests_for_comment/%'
  AND r.rev_timestamp >= '20060101000000'
GROUP BY 1
ORDER BY 1
""", WIKI)

df_volume['year'] = df_volume['ym'].str[:4].astype(int)
df_volume['month'] = df_volume['ym'].str[4:6].astype(int)
df_volume.to_csv(OUT / 'rfc_volume_by_month.csv', index=False)
print(f'{len(df_volume)} months')
df_volume.groupby('year')[['distinct_pages','total_edits']].sum()

## Q1 — Participant tenure in RfC discussions

For each edit to an RfC subpage, compute the editor's account age at time of edit.
This is a proxy for participant tenure (more precise than comment parsing).

Note: `user_registration` is in the `user` table (not `actor`) on Wikimedia replicas.
We join via `actor.actor_user → user.user_id`.

In [ ]:
df_tenure_rfc = wmpaws.run_sql("""
SELECT
    LEFT(r.rev_timestamp, 4)                                AS year,
    r.rev_timestamp                                         AS edit_ts,
    u.user_registration                                     AS reg_ts,
    DATEDIFF(
        STR_TO_DATE(r.rev_timestamp,  '%Y%m%d%H%i%S'),
        STR_TO_DATE(u.user_registration, '%Y%m%d%H%i%S')
    )                                                       AS tenure_days,
    a.actor_name                                            AS username
FROM revision r
JOIN page    p ON r.rev_page    = p.page_id
JOIN actor   a ON r.rev_actor   = a.actor_id
JOIN `user`  u ON a.actor_user  = u.user_id
WHERE p.page_namespace = 4
  AND p.page_title LIKE 'Requests_for_comment/%'
  AND r.rev_timestamp >= '20060101000000'
  AND u.user_registration IS NOT NULL
  AND a.actor_user IS NOT NULL
LIMIT 500000
""", WIKI)

df_tenure_rfc.to_csv(OUT / 'rfc_tenure_raw.csv', index=False)
print(f'{len(df_tenure_rfc):,} rows')
df_tenure_rfc['tenure_days'].describe()

### Tenure by year (trend)

In [ ]:
tenure_by_year = (
    df_tenure_rfc
    .groupby('year')['tenure_days']
    .agg(['median', 'mean', 'count',
          lambda x: x.quantile(0.25),
          lambda x: x.quantile(0.75)])
    .rename(columns={'<lambda_0>': 'p25', '<lambda_1>': 'p75'})
    .reset_index()
)
tenure_by_year.to_csv(OUT / 'rfc_tenure_by_year.csv', index=False)
tenure_by_year

## Q4 — RfA participant tenure (comparison)

Same query as Q1 but for `Requests_for_adminship/%` pages.
Allows direct comparison: are RfA voters more experienced than RfC participants?

In [ ]:
df_tenure_rfa = wmpaws.run_sql("""
SELECT
    LEFT(r.rev_timestamp, 4)                                AS year,
    r.rev_timestamp                                         AS edit_ts,
    u.user_registration                                     AS reg_ts,
    DATEDIFF(
        STR_TO_DATE(r.rev_timestamp,    '%Y%m%d%H%i%S'),
        STR_TO_DATE(u.user_registration,'%Y%m%d%H%i%S')
    )                                                       AS tenure_days,
    a.actor_name                                            AS username
FROM revision r
JOIN page    p ON r.rev_page    = p.page_id
JOIN actor   a ON r.rev_actor   = a.actor_id
JOIN `user`  u ON a.actor_user  = u.user_id
WHERE p.page_namespace = 4
  AND p.page_title LIKE 'Requests_for_adminship/%'
  AND r.rev_timestamp >= '20060101000000'
  AND u.user_registration IS NOT NULL
  AND a.actor_user IS NOT NULL
LIMIT 500000
""", WIKI)

df_tenure_rfa.to_csv(OUT / 'rfa_tenure_raw.csv', index=False)
print(f'{len(df_tenure_rfa):,} rows')
df_tenure_rfa['tenure_days'].describe()

### Side-by-side comparison

In [ ]:
comparison = pd.DataFrame({
    'process': ['RfC', 'RfA'],
    'n': [len(df_tenure_rfc), len(df_tenure_rfa)],
    'median_days': [
        df_tenure_rfc['tenure_days'].median(),
        df_tenure_rfa['tenure_days'].median(),
    ],
    'p25_days': [
        df_tenure_rfc['tenure_days'].quantile(0.25),
        df_tenure_rfa['tenure_days'].quantile(0.25),
    ],
    'p75_days': [
        df_tenure_rfc['tenure_days'].quantile(0.75),
        df_tenure_rfa['tenure_days'].quantile(0.75),
    ],
})
comparison['median_years'] = (comparison['median_days'] / 365).round(1)
comparison.to_csv(OUT / 'rfc_rfa_tenure_comparison.csv', index=False)
comparison

## Q3 — Policy page edit activity

How often are core policy pages edited? Track edits per year to the pages
from our theory corpus — a proxy for how contested/stable policies are.

In [ ]:
POLICY_PAGES = [
    'Requests_for_comment',
    'Consensus',
    'Policies_and_guidelines',
    'Blocking_policy',
    'Banning_policy',
    'Biographies_of_living_persons',
    'Neutral_point_of_view',
    'Verifiability',
    'No_original_research',
    'Civility',
    'Arbitration/Policy',
]

pages_str = ', '.join(f"'{p}'" for p in POLICY_PAGES)

df_policy_edits = wmpaws.run_sql(f"""
SELECT
    p.page_title                       AS page,
    LEFT(r.rev_timestamp, 4)           AS year,
    COUNT(*)                           AS edits,
    COUNT(DISTINCT a.actor_name)       AS distinct_editors
FROM revision r
JOIN page  p ON r.rev_page  = p.page_id
JOIN actor a ON r.rev_actor = a.actor_id
WHERE p.page_namespace = 4
  AND p.page_title IN ({pages_str})
  AND r.rev_timestamp >= '20040101000000'
GROUP BY page, year
ORDER BY page, year
""", WIKI)

df_policy_edits.to_csv(OUT / 'policy_page_edits_by_year.csv', index=False)
print(f'{len(df_policy_edits):,} rows')
df_policy_edits.pivot_table(index='page', columns='year', values='edits', fill_value=0)

## Bonus — Unique participants per RfC (participation scale)

For each RfC subpage, count distinct editors. Gives the distribution of
how many people participate in a typical RfC.

In [ ]:
df_participation = wmpaws.run_sql("""
SELECT
    p.page_title                        AS rfc_page,
    LEFT(MIN(r.rev_timestamp), 4)       AS year_opened,
    COUNT(DISTINCT a.actor_name)        AS distinct_editors,
    COUNT(*)                            AS total_edits,
    MIN(r.rev_timestamp)                AS first_edit,
    MAX(r.rev_timestamp)                AS last_edit,
    DATEDIFF(
        STR_TO_DATE(MAX(r.rev_timestamp), '%Y%m%d%H%i%S'),
        STR_TO_DATE(MIN(r.rev_timestamp), '%Y%m%d%H%i%S')
    )                                   AS duration_days
FROM revision r
JOIN page  p ON r.rev_page  = p.page_id
JOIN actor a ON r.rev_actor = a.actor_id
WHERE p.page_namespace = 4
  AND p.page_title LIKE 'Requests_for_comment/%'
  AND r.rev_timestamp >= '20060101000000'
GROUP BY p.page_id
HAVING total_edits >= 2
ORDER BY year_opened, rfc_page
""", WIKI)

df_participation.to_csv(OUT / 'rfc_participation_per_page.csv', index=False)
print(f'{len(df_participation):,} RfC pages')
df_participation[['distinct_editors','total_edits','duration_days']].describe().round(1)

## Save summary

Quick text summary of what we found — copy this into the project notes.

In [ ]:
lines = [
    '# RfC PAWS Baseline — Summary',
    f'RfC pages in dataset:      {df_participation["rfc_page"].nunique():,}',
    f'Year range:                {df_volume["year"].min()} – {df_volume["year"].max()}',
    f'Median tenure RfC (days):  {df_tenure_rfc["tenure_days"].median():.0f}',
    f'Median tenure RfA (days):  {df_tenure_rfa["tenure_days"].median():.0f}',
    f'Median editors per RfC:    {df_participation["distinct_editors"].median():.1f}',
    f'Median RfC duration (days):{df_participation["duration_days"].median():.1f}',
]
summary = '\n'.join(lines)
print(summary)
(OUT / 'summary.txt').write_text(summary)